# Stage 2 final TEST evaluation (IE643 floor-plan completion) - run once

What this notebook does (everything from **Save Version -> Save & Run All**, no API token, no internet needed):

1. copies the project code from the attached dataset `floorplan-code` and installs the parser / diffusion packages
   offline from the wheels in `floorplan-stage2-final`, then checks their versions;
2. puts the checkpoints from `floorplan-stage2-final` where the code expects them;
3. finds the CubiCasa **test** split (`parser_test.zip`, `completion_test.zip`) in the attached `floorplan-stage2-data`
   output (with `SMOKE = True`: the **val** split instead, 2 cases per section - a code check that never touches test);
4. runs `src/stage2/final_test.py` as **two processes, one per T4 GPU**, cases split by index; every finished case is
   written at once to `/kaggle/working/final_test/part<k>.jsonl` (checkpoint): if the run is cut off, re-run with this
   notebook's previous output attached and it continues where it stopped;
5. writes the report `/kaggle/working/final_test/final_test.md` (FILL: U-Net / constrained / control / shipped large-hole
   method, the constraint ablation with paired CIs; CHECK on test drawings; end to end; all by omission type /
   location / position / size bin) and zips the folder.

Settings: GPU **T4 x2**; Internet may stay off. Sampling = the validated fast settings. Expected time: under 1 h
(about 15 min for `SMOKE = True`).

In [ ]:
import glob, os, shutil, subprocess, sys, time, zipfile
SMOKE = False          # True: 2 cases per section on the VAL split (code check); False: the real test, run once
WORK = "/kaggle/working"
CODE = "/tmp/code"
OUT = f"{WORK}/final_test" + ("_smoke_val" if SMOKE else "")
os.makedirs(OUT, exist_ok=True)

def find(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

# 1. code (the floorplan-code dataset holds code.zip, or its unpacked folder)
hits = [h for h in find("src/utils.py") if "floorplan-code" in h]
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE, dirs_exist_ok=True)
else:
    zipfile.ZipFile([h for h in find("code.zip") if "floorplan-code" in h][0]).extractall(CODE)
print("code ->", CODE, os.listdir(CODE)[:10])

In [ ]:
# 1b. packages, offline from the wheels in floorplan-stage2-final, then a version check
wheels = sorted({os.path.dirname(w) for w in find("wheels/*.whl")})
need = ["segmentation-models-pytorch==0.3.4", "timm==0.9.7", "pretrainedmodels==0.7.4", "efficientnet-pytorch==0.7.1"]
try:
    import diffusers  # noqa: F401  (Kaggle's own diffusers is fine when present)
except ImportError:
    need.append("diffusers==0.30.3")
args = [sys.executable, "-m", "pip", "install", "-q", *need]
if wheels:
    args += ["--no-index", "--find-links", wheels[0]]
print(" ".join(args)); subprocess.run(args, check=True)
import importlib.metadata as md
versions = {p: md.version(p) for p in ("segmentation-models-pytorch", "timm", "diffusers", "torch")}
print(versions)
assert versions["segmentation-models-pytorch"] == "0.3.4" and versions["timm"] == "0.9.7", versions

In [ ]:
# 2. checkpoints: floorplan-stage2-final/checkpoints/<path> -> CODE/<path> (paths of configs/stage2/pipeline.yaml)
import yaml
paths = yaml.safe_load(open(f"{CODE}/configs/stage2/pipeline.yaml", encoding="utf-8"))["checkpoints"]
roots = [h[: -len("checkpoints/" + p)] for name, p in paths.items() for h in find("checkpoints/" + p)]
assert roots, "floorplan-stage2-final (checkpoints) is not attached"
root = roots[0]
for name, p in paths.items():
    src = os.path.join(root, "checkpoints", p)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(f"{CODE}/{p}"), exist_ok=True)
        shutil.copy(src, f"{CODE}/{p}")
        print(f"{name:18s} ok")
    else:
        print(f"{name:18s} MISSING", src)

# 3. test data
split = "val" if SMOKE else "test"
PZ, CZ = find(f"parser_{split}.zip"), find(f"completion_{split}.zip")
assert PZ and CZ, f"parser_{split}.zip / completion_{split}.zip not found: attach the floorplan-stage2-data output"
PZ, CZ = PZ[0], CZ[0]
print(PZ, CZ)

In [ ]:
# 4a. resume: copy the finished cases of an earlier version of this notebook (if its output is attached)
for f in find("final_test/part*.jsonl"):
    dst = f"{OUT}/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst)
        print("resuming from", f, sum(1 for _ in open(f)), "cases")

# 4b. quick GPU check (no test data touched)
import torch
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, "set Accelerator = GPU T4 x2"

In [ ]:
# 4c. the evaluation: one process per GPU
n = max(1, torch.cuda.device_count())
threads = str(max(1, (os.cpu_count() or 2) // n))   # the CPU parts (CHECK, verifier) of both processes share the cores
env = {**os.environ, "PYTHONUTF8": "1", "PYTHONUNBUFFERED": "1", "STAGE2_DEVICE": "cuda", "PYTHONPATH": CODE,
       "OMP_NUM_THREADS": threads, "MKL_NUM_THREADS": threads}
procs = []
for k in range(n):
    log = open(f"{OUT}/part{k}.log", "a")
    cmd = [sys.executable, "-m", "src.stage2.final_test", "--parser-zip", PZ, "--completion-zip", CZ, "--out", OUT,
           "--part", str(k), "--parts", str(n)] + (["--smoke"] if SMOKE else [])
    procs.append(subprocess.Popen(cmd, cwd=CODE, stdout=log, stderr=subprocess.STDOUT,
                                  env={**env, "CUDA_VISIBLE_DEVICES": str(k)}))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    done = sum(sum(1 for _ in open(f)) for f in glob.glob(f"{OUT}/part*.jsonl"))
    print(f"{(time.time() - t0) / 3600:.2f} h, {done} cases done", flush=True)
codes = [p.wait() for p in procs]
print("exit codes", codes)
for k in range(n):
    print(f"===== part{k}.log (tail)"); print("".join(open(f"{OUT}/part{k}.log").readlines()[-20:]))
if any(codes):
    print("!!!!! A SHARD FAILED (exit code != 0): the report below is INCOMPLETE. Fix, then re-run this notebook with "
          "its own output attached: finished cases are kept.")

In [ ]:
# 5. report + zip of all outputs
subprocess.run([sys.executable, "-m", "src.stage2.final_test", "--report", "--out", OUT], cwd=CODE,
               env={**env, "STAGE2_DEVICE": "cpu"}, check=True)
shutil.make_archive(f"{WORK}/final_test", "zip", OUT)
print(open(f"{OUT}/final_test.md", encoding="utf-8").read())